# Pandalyzer — Manual Validator Tests
## Purpose
Manually test required-column, missingness, dtype, and numeric-range validation. Synthetic DataFrames make the expected outcomes deterministic.

## 1. Imports

In [1]:
import sys
from pathlib import Path
import json
import pandas as pd

_candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (p for p in _candidates if (p / "database").is_dir() and (p / "src" / "autoclean").is_dir()),
    Path.cwd(),
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from autoclean.cleaner import remove_duplicate_rows, handle_missing_values, clean_dataframe
from autoclean.profiler import profile_dataframe, compare_data_quality
from autoclean.validators import validate_missing_values, validate_required_columns, validate_column_types, validate_numeric_ranges
from autoclean.connectors import load_csv, save_csv, save_quality_report

csv_path = PROJECT_ROOT / "database" / "messy_retail_orders.csv"
print("Project root:", PROJECT_ROOT)
print("Dataset available:", csv_path.exists())

Project root: /run/media/amateur/MATRIX/Pandalyzer/AutoClean
Dataset available: True


## 2. Required-column validation
The function should return only required columns that are absent.

In [2]:
test_df = pd.DataFrame({"order_id":[1,2], "amount":[10.0,20.0]})
required = ["order_id","amount","customer_id"]
missing_required = validate_required_columns(test_df, required)
print(missing_required)
assert missing_required == ["customer_id"]
assert validate_required_columns(test_df, ["order_id"]) == []
print("PASS: required-column validation")

['customer_id']
PASS: required-column validation


## 3. Missing-value threshold validation
A column is returned only when its missing percentage is strictly greater than the threshold.

In [3]:
missing_df = pd.DataFrame({"low":[1,None,3,4], "high":[None,None,3,None], "none":[1,2,3,4]})
flagged = validate_missing_values(missing_df, threshold=50)
print("Flagged:", flagged)
assert flagged == ["high"]
assert validate_missing_values(missing_df, threshold=75) == []
print("PASS: missing-value threshold validation")

Flagged: ['high']
PASS: missing-value threshold validation


## 4. Column-type validation
Check correct dtypes, a dtype mismatch, and a missing column.

In [4]:
typed_df = pd.DataFrame({"quantity":pd.Series([1,2],dtype="int64"), "amount":pd.Series([10.0,20.0],dtype="float64")})
problems = validate_column_types(typed_df, {"quantity":"object","amount":"float64","customer_id":"object"})
display(pd.Series(problems, name="problem").to_frame())
assert problems["customer_id"] == "missing column"
assert "expected object" in problems["quantity"]
assert "amount" not in problems
print("PASS: dtype validation")

,problem
quantity,"expected object, found int64"
customer_id,missing column


PASS: dtype validation


## 5. Numeric-range validation
Check lower/upper bounds, missing values, and returned index labels.

In [5]:
range_df = pd.DataFrame({"quantity":[1,0,5,101,None], "discount_pct":[0,10,50,101,None], "rating":[1,3,5,6,None]})
problems = validate_numeric_ranges(range_df, {"quantity":(1,100),"discount_pct":(0,100),"rating":(1,5)})
display(pd.Series(problems, name="invalid index labels").to_frame())
assert problems == {"quantity":[1,3], "discount_pct":[3], "rating":[3]}
print("PASS: numeric-range validation")

,invalid index labels
quantity,"[1, 3]"
discount_pct,[3]
rating,[3]


PASS: numeric-range validation


## 6. Real-dataset schema and missingness check
Inspect the actual CSV without modifying it.

In [6]:
if csv_path.exists():
    real_df = load_csv(csv_path)
    expected = ["order_id","order_date","customer_id","product_id","quantity","unit_price","discount_pct","order_amount","payment_method","order_status"]
    print("Missing required columns:", validate_required_columns(real_df, expected))
    print("Columns over 20% missing:", validate_missing_values(real_df, threshold=20))
    print("Dtypes:")
    display(real_df.dtypes.rename("dtype").to_frame())
else:
    print("CSV not available; synthetic validator tests above still ran.")

Missing required columns: []
Columns over 20% missing: ['coupon_code', 'notes']
Dtypes:


,dtype
order_id,str
order_date,str
customer_id,str
product_id,str
category,str
quantity,str
unit_price,str
discount_pct,str
order_amount,str
payment_method,str
